In [22]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

# from pathlib import Path
import joblib

In [23]:

# Load dataset
DATA_PATH = Path("../data/processed/terai_features_engineered.parquet")
df = pd.read_parquet(DATA_PATH)
df.head(3)


,Date,Year,Day_of_Year,Province,District,Regional_Hub,Latitude,Longitude,Max_Temp_C,Min_Temp_C,...,DOY_Sin,DOY_Cos,Month_Sin,Month_Cos,Season,Rain_Log,Yesterday_Rain_Log,Past_7D_Rain_Log,Is_Western_Terai,Terai_Region
0,2000-01-15,2000,15,Lumbini,Banke,Nepalgunj,28.05,81.62,25.11,10.84,...,0.2552,0.9669,0.5,0.866,Winter,0.0,0.0,2.3915,1,Western Terai
1,2000-01-16,2000,16,Lumbini,Banke,Nepalgunj,28.05,81.62,23.06,7.54,...,0.2718,0.9624,0.5,0.866,Winter,0.0,0.0,2.3915,1,Western Terai
2,2000-01-17,2000,17,Lumbini,Banke,Nepalgunj,28.05,81.62,21.70,4.68,...,0.2883,0.9575,0.5,0.866,Winter,0.0,0.0,2.3897,1,Western Terai


In [24]:
df.info()
# df.columns

<class 'pandas.DataFrame'>
RangeIndex: 204582 entries, 0 to 204581
Data columns (total 65 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Date                  204582 non-null  datetime64[ns]
 1   Year                  204582 non-null  int32         
 2   Day_of_Year           204582 non-null  int32         
 3   Province              204582 non-null  category      
 4   District              204582 non-null  category      
 5   Regional_Hub          204582 non-null  category      
 6   Latitude              204582 non-null  float64       
 7   Longitude             204582 non-null  float64       
 8   Max_Temp_C            204582 non-null  float64       
 9   Min_Temp_C            204582 non-null  float64       
 10  Mean_Temp_C           204582 non-null  float64       
 11  Humidity_Pct          204582 non-null  float64       
 12  Wind_Speed_ms         204582 non-null  float64       
 13  Precipitat

In [25]:
# target columns labels 
# y_next_day = df["Next_Day_Max_Temp"]
# y_heat_flag = df["Heatwave_Flag"]
# y_heat_level = df["Heatwave_Severity"]
y_next_day = df["Next_Day_Max_Temp"]
y_heat_flag = (df["Next_Day_Max_Temp"] >= 40.0).astype(int)
conditions = [
    df["Next_Day_Max_Temp"] < 40.0,
    (df["Next_Day_Max_Temp"] >= 40.0) & (df["Next_Day_Max_Temp"] < 42.0),
    (df["Next_Day_Max_Temp"] >= 42.0) & (df["Next_Day_Max_Temp"] < 45.0),
    df["Next_Day_Max_Temp"] >= 45.0
]
choices = [0, 1, 2, 3]
y_heat_level = pd.Series(np.select(conditions, choices, default=0), index=df.index, dtype="int8")



In [26]:
# decoupled
meta_cols = ["Date", "District", "Province", "Regional_Hub"]
df_meta = df[meta_cols].copy()

df_meta.sample(3)


,Date,District,Province,Regional_Hub
35756,2017-12-01,Chitwan,Bagmati,Bharatpur
69839,2004-07-17,Kailali,Sudurpashchim,Dhangadhi
194452,2025-08-25,Siraha,Madhesh,Lahan


In [27]:
# encode
cat_cols = ["Season", "Terai_Region"]
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype="int8")

dummy_cols = [c for c in df_encoded.columns if c.startswith("Season_") or c.startswith("Terai_Region_")]
df_encoded[dummy_cols].sample(3)


,Season_Monsoon,Season_Post-Monsoon,Season_Pre-Monsoon,Season_Winter,Terai_Region_Central Terai,Terai_Region_Eastern Terai,Terai_Region_Western Terai
21898,1,0,0,0,0,0,1
77035,0,0,1,0,0,0,1
203279,0,0,0,1,0,1,0


In [28]:
# 
target_cols = ["Next_Day_Max_Temp", "Heatwave_Flag", "Heatwave_Severity"]
drop_cols = meta_cols + target_cols
X = df_encoded.drop(columns=drop_cols)
X.sample(3)

,Year,Day_of_Year,Latitude,Longitude,Max_Temp_C,Min_Temp_C,Mean_Temp_C,Humidity_Pct,Wind_Speed_ms,Precipitation_mm,...,Yesterday_Rain_Log,Past_7D_Rain_Log,Is_Western_Terai,Season_Monsoon,Season_Post-Monsoon,Season_Pre-Monsoon,Season_Winter,Terai_Region_Central Terai,Terai_Region_Eastern Terai,Terai_Region_Western Terai
122948,2016,215,27.65,84.12,31.08,24.81,27.48,85.62,1.08,1.35,...,1.9125,4.5682,0,1,0,0,0,1,0,0
104601,2019,256,26.64,85.80,32.45,26.17,28.55,85.04,3.20,12.87,...,1.9530,3.4819,0,1,0,0,0,0,1,0
112064,2013,168,26.45,87.27,35.11,25.59,30.22,68.91,1.26,3.42,...,1.9643,3.1390,0,1,0,0,0,0,1,0


In [29]:
# train_val_test
train_mask = df_meta["Date"] <= "2020-12-31"
val_mask   = df_meta["Date"].between("2021-01-01", "2022-12-31")
test_mask  = df_meta["Date"] >= "2023-01-01"

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
y_train_flag, y_val_flag, y_test_flag = y_heat_flag[train_mask], y_heat_flag[val_mask], y_heat_flag[test_mask]
y_train_mtemp, y_val_mtemp, y_test_mtemp = y_next_day[train_mask], y_next_day[val_mask], y_next_day[test_mask]
y_train_sev,  y_val_sev,  y_test_sev  = y_heat_level[train_mask], y_heat_level[val_mask], y_heat_level[test_mask]


In [30]:
# Step 07: Target Distribution & Class Imbalance Audit
print(f"Train Heatwave Rate: {y_train_flag.mean()*100:.2f}% ({y_train_flag.sum():,} events)")
print(f"Val Heatwave Rate  : {y_val_flag.mean()*100:.2f}% ({y_val_flag.sum():,} events)")
print(f"Test Heatwave Rate : {y_test_flag.mean()*100:.2f}% ({y_test_flag.sum():,} events)")

# Negative-to-positive ratio for cost-sensitive loss weighting
pos_weight = (len(y_train_flag) - y_train_flag.sum()) / y_train_flag.sum()
print(f"scale_pos_weight ratio (Neg:Pos) = {pos_weight:.2f}:1")


Train Heatwave Rate: 6.80% (10,932 events)
Val Heatwave Rate  : 3.05% (468 events)
Test Heatwave Rate : 6.34% (1,804 events)
scale_pos_weight ratio (Neg:Pos) = 13.71:1


In [31]:
# fitting and saving
Path("../models").mkdir(parents=True, exist_ok=True)
scaler = StandardScaler()
scaler.fit(X_train)
joblib.dump(scaler, "../models/preprocessing.pkl")
print("✅ ../models/preprocessing.pkl saved")


✅ ../models/preprocessing.pkl saved


In [32]:
# scalling
X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_val_scaled shape  : {X_val_scaled.shape}")
print(f"X_test_scaled shape : {X_test_scaled.shape}")


X_train_scaled shape: (160797, 63)
X_val_scaled shape  : (15330, 63)
X_test_scaled shape : (28455, 63)


In [33]:
# ==============================================================================
# Step 10: Setup Temporal Walk-Forward Cross-Validation (TimeSeriesSplit)
# ==============================================================================

# 1. Get true chronological sort order by Date and District
sort_idx = df_meta[train_mask].sort_values(by=["Date", "District"]).index

# 2. Re-order X_train, scaled arrays, and targets to strictly respect time
X_train_chrono = X_train.loc[sort_idx].reset_index(drop=True)
y_train_flag_chrono = y_train_flag.loc[sort_idx].reset_index(drop=True)
y_train_sev_chrono  = y_train_sev.loc[sort_idx].reset_index(drop=True)
y_train_mtemp_chrono = y_train_mtemp.loc[sort_idx].reset_index(drop=True)

# Re-scale in true chronological order
X_train_scaled_chrono = scaler.transform(X_train_chrono)

# 3. Setup 5-Fold Expanding Window TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

# Verify the temporal boundaries of each fold
print("--- 5-Fold Temporal Walk-Forward Splits (Train 2000–2020) ---")
dates_train = df_meta.loc[sort_idx, "Date"].reset_index(drop=True)

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train_chrono), 1):
    tr_start = dates_train.iloc[tr_idx[0]].date()
    tr_end   = dates_train.iloc[tr_idx[-1]].date()
    va_start = dates_train.iloc[va_idx[0]].date()
    va_end   = dates_train.iloc[va_idx[-1]].date()
    
    print(f"Fold {fold}: Train [{tr_start} to {tr_end}] ({len(tr_idx):,} rows) "
          f"-> Val [{va_start} to {va_end}] ({len(va_idx):,} rows)")


--- 5-Fold Temporal Walk-Forward Splits (Train 2000–2020) ---
Fold 1: Train [2000-01-15 to 2003-07-14] (26,802 rows) -> Val [2003-07-14 to 2007-01-10] (26,799 rows)
Fold 2: Train [2000-01-15 to 2007-01-10] (53,601 rows) -> Val [2007-01-10 to 2010-07-09] (26,799 rows)
Fold 3: Train [2000-01-15 to 2010-07-09] (80,400 rows) -> Val [2010-07-09 to 2014-01-05] (26,799 rows)
Fold 4: Train [2000-01-15 to 2014-01-05] (107,199 rows) -> Val [2014-01-05 to 2017-07-04] (26,799 rows)
Fold 5: Train [2000-01-15 to 2017-07-04] (133,998 rows) -> Val [2017-07-04 to 2020-12-31] (26,799 rows)


### Init models 


In [34]:
# ==============================================================================
# Unified Imports: Models, Metrics & Scoring
# ==============================================================================
import time
import numpy as np
import pandas as pd

# --- 1. Candidate Models by Family ---
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import Lasso, LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

# --- 2. Classification Evaluation Metrics (Target 1 & Target 2) ---
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,  # PR-AUC
    brier_score_loss,  # Calibration
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# --- 3. Regression Evaluation Metrics (Target 3) ---
from sklearn.metrics import (
    max_error,
    mean_absolute_error,  # MAE (°C)
    mean_squared_error,  # RMSE (°C)
    r2_score,
)

# --- 4. Shared Negative-to-Positive Imbalance Ratio (~14.5:1) ---
pos_weight = (len(y_train_flag) - y_train_flag.sum()) / y_train_flag.sum()


In [35]:
models_target1_flag = {
    # 1. Linear Family (Requires scaled features, Cost-Sensitive)
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    # 2. Tree Bagging Family (Unscaled features, Parallelized)
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42
    ),
    # 3. Tree Boosting Family
    "HistGradientBoosting": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=42
    ),
    "LightGBM": LGBMClassifier(
        scale_pos_weight=pos_weight,
        n_estimators=150,
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        scale_pos_weight=pos_weight,
        n_estimators=150,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    ),
}


models_target2_severity = {
    # 1. Linear Family (Automatically handles multinomial, Scaled)
    "Logistic Regression (Multinomial)": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    # 2. Tree Bagging Family (Unscaled features, Parallelized)
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42
    ),
    # 3. Tree Boosting Family (HistGradientBoosting Multi-Class)
    "HistGradientBoosting": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=42
    ),
    # 4. Tree Boosting Family (LightGBM Multi-Class)
    "LightGBM": LGBMClassifier(
        objective="multiclass",
        num_class=4,
        class_weight="balanced",
        n_estimators=150,
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),
    # 5. Tree Boosting Family (XGBoost Multi-Class)
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        n_estimators=150,
        random_state=42,
        eval_metric="mlogloss",
        n_jobs=-1,
    ),
}

models_target3_temp = {
    # 1. Linear Family (L2 Regularized Baseline, Scaled)
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    # 1. Linear Family (L1 Sparse Regularized, Scaled)
    "Lasso Regression": Lasso(alpha=0.1, random_state=42),
    # 2. Tree Bagging Family (Unscaled features, Parallelized)
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100, n_jobs=-1, random_state=42
    ),
    # 3. Tree Boosting Family
    "HistGradientBoosting Regressor": HistGradientBoostingRegressor(
        random_state=42
    ),
    "LightGBM Regressor": LGBMRegressor(
        n_estimators=150, random_state=42, verbosity=-1, n_jobs=-1
    ),
    "XGBoost Regressor": XGBRegressor(
        n_estimators=150, random_state=42, n_jobs=-1
    ),
}




In [36]:
# ==============================================================================
# Batch Training & Validation Evaluation Loop: Target 1 (Heatwave_Flag)
# ==============================================================================
import time
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

target1_results = []

print("=== Training & Evaluating Target 1 Models on Validation (2021–2022) ===")

for name, model in models_target1_flag.items():
  t0 = time.time()

  # Route scaled data to linear models, unscaled to tree-based models
  X_tr = X_train_scaled if "Logistic" in name else X_train
  X_va = X_val_scaled if "Logistic" in name else X_val

  # 1. Fit on Training Partition (2000–2020)
  print(f"Training {name}...", end=" ")
  model.fit(X_tr, y_train_flag)
  train_time = time.time() - t0

  # 2. Predict on Validation Partition (2021–2022)
  val_preds = model.predict(X_va)
  val_probs = model.predict_proba(X_va)[:, 1]

  # 3. Calculate Key Climate DRR Metrics
  prec = precision_score(y_val_flag, val_preds, zero_division=0)
  rec = recall_score(y_val_flag, val_preds, zero_division=0)
  f1 = f1_score(y_val_flag, val_preds, zero_division=0)
  roc = roc_auc_score(y_val_flag, val_probs)
  pr_auc = average_precision_score(y_val_flag, val_probs)
  brier = brier_score_loss(y_val_flag, val_probs)

  target1_results.append({
      "Model": name,
      "Train Time (s)": round(train_time, 2),
      "Precision": round(prec, 4),
      "Recall": round(rec, 4),
      "F1-Score": round(f1, 4),
      "PR-AUC": round(pr_auc, 4),
      "ROC-AUC": round(roc, 4),
      "Brier Score": round(brier, 4),
  })
  print(
      f"Done ({train_time:.1f}s) | Recall: {rec:.4f} | Prec: {prec:.4f} | F1:"
      f" {f1:.4f} | PR-AUC: {pr_auc:.4f}"
  )

# Display Ranked Leaderboard
df_target1_leaderboard = (
    pd.DataFrame(target1_results)
    .sort_values(by="PR-AUC", ascending=False)
    .reset_index(drop=True)
)
print("\n" + "=" * 80)
print("🎯 TARGET 1 VALIDATION LEADERBOARD (Ranked by PR-AUC)")
print("=" * 80)
display(df_target1_leaderboard)


=== Training & Evaluating Target 1 Models on Validation (2021–2022) ===
Training Logistic Regression... Done (1.7s) | Recall: 0.9722 | Prec: 0.3748 | F1: 0.5410 | PR-AUC: 0.7656
Training Random Forest... Done (3.1s) | Recall: 0.6923 | Prec: 0.6291 | F1: 0.6592 | PR-AUC: 0.7353
Training HistGradientBoosting... Done (20.2s) | Recall: 0.8590 | Prec: 0.4914 | F1: 0.6252 | PR-AUC: 0.7346
Training LightGBM... Done (1.3s) | Recall: 0.7564 | Prec: 0.5160 | F1: 0.6135 | PR-AUC: 0.6946
Training XGBoost... Done (2.0s) | Recall: 0.6474 | Prec: 0.6352 | F1: 0.6413 | PR-AUC: 0.7047

🎯 TARGET 1 VALIDATION LEADERBOARD (Ranked by PR-AUC)


,Model,Train Time (s),Precision,Recall,F1-Score,PR-AUC,ROC-AUC,Brier Score
0,Logistic Regression,1.70,0.3748,0.9722,0.5410,0.7656,0.9899,0.0354
1,Random Forest,3.11,0.6291,0.6923,0.6592,0.7353,0.9871,0.0152
2,HistGradientBoosting,20.20,0.4914,0.8590,0.6252,0.7346,0.9881,0.0212
3,XGBoost,1.97,0.6352,0.6474,0.6413,0.7047,0.9844,0.0177
4,LightGBM,1.31,0.5160,0.7564,0.6135,0.6946,0.9809,0.0207


In [37]:
# ==============================================================================
# Batch Training & Validation Evaluation Loop: Target 2 (Heatwave_Severity)
# Tiers: Tier 0 (<40°C), Tier 1 (40–41.9°C), Tier 2 (42–44.9°C), Tier 3 (>=45°C)
# ==============================================================================
import time
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    recall_score,
    roc_auc_score,
)

target2_results = []

print(
    "=== Training & Evaluating Target 2 (Severity) Models on Validation"
    " (2021–2022) ==="
)

for name, model in models_target2_severity.items():
  t0 = time.time()

  # Route scaled data to linear models, unscaled to tree-based models
  X_tr = X_train_scaled if "Logistic" in name else X_train
  X_va = X_val_scaled if "Logistic" in name else X_val

  # 1. Fit on Training Partition (2000–2020)
  print(f"Training {name}...", end=" ")
  model.fit(X_tr, y_train_sev)
  train_time = time.time() - t0

  # 2. Predict classes and probabilities on Validation Partition (2021–2022)
  val_preds = model.predict(X_va)
  val_probs = model.predict_proba(X_va)

  # 3. Calculate Multi-Class & Per-Tier Metrics
  acc = accuracy_score(y_val_sev, val_preds)
  macro_f1 = f1_score(y_val_sev, val_preds, average="macro", zero_division=0)
  weighted_f1 = f1_score(
      y_val_sev, val_preds, average="weighted", zero_division=0
  )

  # Per-tier recall (Disaster Risk Reduction priority)
  tier_recalls = recall_score(
      y_val_sev, val_preds, average=None, zero_division=0
  )
  rec_t0 = tier_recalls[0] if len(tier_recalls) > 0 else 0
  rec_t1 = tier_recalls[1] if len(tier_recalls) > 1 else 0
  rec_t2 = tier_recalls[2] if len(tier_recalls) > 2 else 0
  rec_t3 = tier_recalls[3] if len(tier_recalls) > 3 else 0

  # Multi-class ROC-AUC (One-vs-Rest)
  try:
    roc_ovr = roc_auc_score(
        y_val_sev, val_probs, multi_class="ovr", average="macro"
    )
  except Exception:
    roc_ovr = float("nan")

  target2_results.append({
      "Model": name,
      "Train Time (s)": round(train_time, 2),
      "Accuracy": round(acc, 4),
      "Macro F1": round(macro_f1, 4),
      "Weighted F1": round(weighted_f1, 4),
      "Tier 0 Recall": round(rec_t0, 4),
      "Tier 1 Recall": round(rec_t1, 4),
      "Tier 2 Recall": round(rec_t2, 4),
      "Tier 3 Recall": round(rec_t3, 4),
      "ROC-AUC (OvR)": round(roc_ovr, 4),
  })

  print(
      f"Done ({train_time:.1f}s) | Macro F1: {macro_f1:.4f} | Tier 2 Rec:"
      f" {rec_t2:.4f} | Tier 3 Rec: {rec_t3:.4f}"
  )

# Display Ranked Leaderboard
df_target2_leaderboard = (
    pd.DataFrame(target2_results)
    .sort_values(by="Macro F1", ascending=False)
    .reset_index(drop=True)
)
print("\n" + "=" * 90)
print("🎯 TARGET 2 (SEVERITY) VALIDATION LEADERBOARD (Ranked by Macro F1)")
print("=" * 90)
display(df_target2_leaderboard)


=== Training & Evaluating Target 2 (Severity) Models on Validation (2021–2022) ===
Training Logistic Regression (Multinomial)... 

e:\installexe\Anaconda\envs\DS\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Done (20.1s) | Macro F1: 0.4810 | Tier 2 Rec: 0.5806 | Tier 3 Rec: 0.5000
Training Random Forest... Done (2.2s) | Macro F1: 0.5310 | Tier 2 Rec: 0.6559 | Tier 3 Rec: 0.0000
Training HistGradientBoosting... Done (17.3s) | Macro F1: 0.5268 | Tier 2 Rec: 0.6559 | Tier 3 Rec: 0.5000
Training LightGBM... Done (3.9s) | Macro F1: 0.5710 | Tier 2 Rec: 0.6774 | Tier 3 Rec: 0.5000
Training XGBoost... Done (7.2s) | Macro F1: 0.6062 | Tier 2 Rec: 0.6882 | Tier 3 Rec: 0.5000

🎯 TARGET 2 (SEVERITY) VALIDATION LEADERBOARD (Ranked by Macro F1)


,Model,Train Time (s),Accuracy,Macro F1,Weighted F1,Tier 0 Recall,Tier 1 Recall,Tier 2 Recall,Tier 3 Recall,ROC-AUC (OvR)
0,XGBoost,7.19,0.9753,0.6062,0.9717,0.9952,0.2574,0.6882,0.5,0.9900
1,LightGBM,3.91,0.9622,0.5710,0.9664,0.9752,0.5147,0.6774,0.5,0.9869
2,Random Forest,2.22,0.9698,0.5310,0.9726,0.9804,0.6300,0.6559,0.0,0.9891
3,HistGradientBoosting,17.25,0.9485,0.5268,0.9597,0.9557,0.7373,0.6559,0.5,0.9880
4,Logistic Regression (Multinomial),20.06,0.9362,0.4810,0.9525,0.9412,0.8284,0.5806,0.5,0.9859


In [38]:
# ==============================================================================
# Batch Training & Validation Evaluation Loop: Target 3 (Next_Day_Max_Temp)
# ==============================================================================
import time
import numpy as np
import pandas as pd
from sklearn.metrics import max_error, mean_absolute_error, mean_squared_error, r2_score

target3_results = []

print(
    "=== Training & Evaluating Target 3 (Temperature) Models on Validation"
    " (2021–2022) ==="
)

for name, model in models_target3_temp.items():
  t0 = time.time()

  # Route scaled data to linear models (Ridge/Lasso), unscaled to tree-based models
  is_linear = any(term in name for term in ["Ridge", "Lasso", "Linear"])
  X_tr = X_train_scaled if is_linear else X_train
  X_va = X_val_scaled if is_linear else X_val

  # 1. Fit on Training Partition (2000–2020)
  print(f"Training {name}...", end=" ")
  model.fit(X_tr, y_train_mtemp)
  train_time = time.time() - t0

  # 2. Predict continuous temperature on Validation Partition (2021–2022)
  val_preds = model.predict(X_va)

  # 3. Calculate Temperature Forecasting Metrics
  mae = mean_absolute_error(y_val_mtemp, val_preds)
  rmse = np.sqrt(mean_squared_error(y_val_mtemp, val_preds))
  r2 = r2_score(y_val_mtemp, val_preds)
  worst_miss = max_error(y_val_mtemp, val_preds)

  target3_results.append({
      "Model": name,
      "Train Time (s)": round(train_time, 2),
      "MAE (°C)": round(mae, 4),
      "RMSE (°C)": round(rmse, 4),
      "R² Score": round(r2, 4),
      "Max Error (°C)": round(worst_miss, 2),
  })

  print(
      f"Done ({train_time:.1f}s) | MAE: {mae:.2f}°C | RMSE: {rmse:.2f}°C | R²:"
      f" {r2:.4f}"
  )

# Display Ranked Leaderboard (Lowest MAE is #1)
df_target3_leaderboard = (
    pd.DataFrame(target3_results)
    .sort_values(by="MAE (°C)", ascending=True)
    .reset_index(drop=True)
)
print("\n" + "=" * 80)
print("🎯 TARGET 3 (TEMPERATURE) VALIDATION LEADERBOARD (Ranked by Lowest MAE)")
print("=" * 80)
display(df_target3_leaderboard)


=== Training & Evaluating Target 3 (Temperature) Models on Validation (2021–2022) ===
Training Ridge Regression... Done (0.1s) | MAE: 0.95°C | RMSE: 1.30°C | R²: 0.9487
Training Lasso Regression... Done (0.6s) | MAE: 1.01°C | RMSE: 1.37°C | R²: 0.9429
Training Random Forest Regressor... Done (56.9s) | MAE: 0.96°C | RMSE: 1.30°C | R²: 0.9484
Training HistGradientBoosting Regressor... Done (1.5s) | MAE: 0.97°C | RMSE: 1.30°C | R²: 0.9484
Training LightGBM Regressor... Done (1.6s) | MAE: 0.97°C | RMSE: 1.31°C | R²: 0.9476
Training XGBoost Regressor... Done (1.7s) | MAE: 1.07°C | RMSE: 1.45°C | R²: 0.9363

🎯 TARGET 3 (TEMPERATURE) VALIDATION LEADERBOARD (Ranked by Lowest MAE)


,Model,Train Time (s),MAE (°C),RMSE (°C),R² Score,Max Error (°C)
0,Ridge Regression,0.13,0.9482,1.2978,0.9487,8.76
1,Random Forest Regressor,56.91,0.9564,1.3018,0.9484,8.69
2,HistGradientBoosting Regressor,1.53,0.9652,1.3023,0.9484,8.62
3,LightGBM Regressor,1.62,0.9715,1.3117,0.9476,8.92
4,Lasso Regression,0.59,1.0072,1.3691,0.9429,9.63
5,XGBoost Regressor,1.69,1.0694,1.4464,0.9363,9.53
